# Introduction to Django REST Framework (DRF)

## What is an API?

An API (Application Programming Interface) is a contract that lets two programs exchange data. The client never touches the database directly — it communicates through the API.

## What is REST?

REST (Representational State Transfer) is a design style for APIs built on top of HTTP.

| HTTP Method | Action | Example Route |
|-------------|--------|---------------|
| GET | Read list | `/books/` |
| POST | Create | `/books/` |
| GET | Read one | `/books/3/` |
| PUT / PATCH | Update | `/books/3/` |
| DELETE | Delete | `/books/3/` |


## JSON vs XML

Modern APIs use JSON. It is lighter and works natively with Python and JavaScript.

**JSON:**
```json
{ "title": "Harry Potter", "author": "J.K. Rowling" }
```

**XML:**
```xml
<book>
  <title>Harry Potter</title>
  <author>J.K. Rowling</author>
</book>
```


## Why DRF Instead of Vanilla Django?

Plain Django views have two problems for APIs:
1. HTTP methods must be separated manually with `if/else`.
2. CSRF protection blocks POST requests from external tools like Postman.

| Feature | Django | DRF |
|---------|--------|-----|
| HTML page rendering | Yes | No |
| JSON input/output | Manual | Automatic |
| Browsable API | No | Built-in |
| Method mapping | `if/else` | `@api_view`, CBVs |
| Auth & pagination | Manual | Built-in |


## Installing and Configuring DRF

```bash
pip install djangorestframework
```

```python
# settings.py
INSTALLED_APPS = [
    ...
    'rest_framework',
]
```


## Hello API with @api_view

```python
# views.py
from rest_framework.decorators import api_view
from rest_framework.response import Response

@api_view(['GET'])
def hello_api(request):
    return Response({'message': 'Hello, API!'})
```

```python
# urls.py
from django.urls import path
from .views import hello_api

urlpatterns = [
    path('hello/', hello_api),
]
```


## DRF Serializers

Serializers convert model instances to JSON and validate incoming data.

### Manual Serializer
```python
from rest_framework import serializers

class BookSerializer(serializers.Serializer):
    id = serializers.IntegerField(read_only=True)
    title = serializers.CharField()
    author = serializers.CharField()
```

### ModelSerializer (recommended)
```python
class BookModelSerializer(serializers.ModelSerializer):
    class Meta:
        model = Book
        fields = ['id', 'title', 'author']
```


## Full CRUD with Function-Based Views

```python
@api_view(['GET', 'POST'])
def book_list(request):
    if request.method == 'GET':
        books = Book.objects.all()
        return Response(BookModelSerializer(books, many=True).data)
    serializer = BookModelSerializer(data=request.data)
    if serializer.is_valid():
        serializer.save()
        return Response(serializer.data, status=201)
    return Response(serializer.errors, status=400)

@api_view(['GET', 'PUT', 'PATCH', 'DELETE'])
def book_detail(request, pk):
    try:
        book = Book.objects.get(pk=pk)
    except Book.DoesNotExist:
        return Response(status=404)
    if request.method == 'GET':
        return Response(BookModelSerializer(book).data)
    elif request.method in ['PUT', 'PATCH']:
        s = BookModelSerializer(book, data=request.data, partial=(request.method == 'PATCH'))
        if s.is_valid():
            s.save()
            return Response(s.data)
        return Response(s.errors, status=400)
    book.delete()
    return Response(status=204)
```

```python
# urls.py
urlpatterns = [
    path('books/', book_list),
    path('books/<int:pk>/', book_detail),
]
```


## Summary

- REST APIs use HTTP methods (GET, POST, PUT, PATCH, DELETE) to perform CRUD.
- JSON is the standard data format for modern APIs.
- DRF solves method separation, CSRF, JSON handling, and browsable API out of the box.
- Serializers convert between Python objects and JSON, and handle validation.
- `ModelSerializer` generates field definitions automatically from a model.
- `@api_view` is the simplest way to create an API endpoint.
